In [0]:
%pip install confluent-kafka
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.0 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 1.7/4.0 MB 51.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 4.0/4.0 MB 71.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 44.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
bootstrap = "pkc-56d1g.eastus.azure.confluent.cloud:9092"
api_key = "CSCAGFSZ6YL7Q6TT"  
api_secret = "cfltIzcaPWEJQq4mS1DxAa6Dr0O7aRXhXr/X6+0aZfHuDJ0tJsLBZM4z3Yv2zWeg"
topic = "orders.events"


In [0]:
from confluent_kafka import Consumer
import json

# Suas credenciais (use dbutils.secrets se configurado)
bootstrap = "pkc-56d1g.eastus.azure.confluent.cloud:9092"
api_key = "CSCAGFSZ6YL7Q6TT"
api_secret = "cfltIzcaPWEJQq4mS1DxAa6Dr0O7aRXhXr/X6+0aZfHuDJ0tJsLBZM4z3Yv2zWeg"
topic = "orders.events"

consumer = Consumer({
    'bootstrap.servers': bootstrap,
    'group.id': 'connect-lcc',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'auto.offset.reset': 'earliest',  # Lê do início
    'enable.auto.commit': True
})

consumer.subscribe([topic])
print(f"🔍 Consumindo '{topic}'... (5 mensagens ou Ctrl+Enter)")

count = 0
try:
    while count < 5:  # Para depois de 5 msgs
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            print(f"❌ Erro: {msg.error()}")
            break
        else:
            value = msg.value().decode('utf-8')
            print(f"✅ Mensagem {count+1}: {json.dumps(json.loads(value), indent=2)}")
            count += 1
except Exception as e:
    print(f"Erro: {e}")
finally:
    consumer.close()
    print("Consumer fechado")

🔍 Consumindo 'orders.events'... (5 mensagens ou Ctrl+Enter)


%6|1775830393.751|GETSUBSCRIPTIONS|rdkafka#consumer-1| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to nQQHbK8qRD2LgtPRwXytKA


✅ Mensagem 1: {
  "order_id": "ord_00001",
  "customer_id": "cust_003",
  "amount": 115.42,
  "currency": "BRL",
  "status": "created",
  "items": [
    {
      "sku": "gloves_pro",
      "qty": 1,
      "unit_price": 115.42
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:04:54.316808+00:00"
}
✅ Mensagem 2: {
  "order_id": "ord_00002",
  "customer_id": "cust_005",
  "amount": 791.82,
  "currency": "BRL",
  "status": "shipped",
  "items": [
    {
      "sku": "gloves_pro",
      "qty": 2,
      "unit_price": 395.91
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:04:58.412134+00:00"
}
✅ Mensagem 3: {
  "order_id": "ord_00003",
  "customer_id": "cust_013",
  "amount": 439.8,
  "currency": "BRL",
  "status": "created",
  "items": [
    {
      "sku": "helmet_red",
      "qty": 3,
      "unit_price": 146.6
    }
  ],
  "source": "api",
  "event_time": "2026-04-10T14:05:00.543566+00:00"
}
✅ Mensagem 4: {
  "order_id": "ord_00004",
  "customer_id": "cust_011",


In [0]:
from confluent_kafka import Consumer, Producer
import time, json, uuid

# Consumer LATEST (só mensagens NOVAS)
consumer = Consumer({
    'bootstrap.servers': bootstrap,
    'group.id': 'connect-lcc',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'auto.offset.reset': 'latest',  # ← SÓ NOVAS!

})
consumer.subscribe(['orders.events'])

print("⏳ Aguardando 10s por novas mensagens...")

msgs = []
start = time.time()
while time.time() - start < 10:
    msg = consumer.poll(1.0)
    if msg and not msg.error():
        msgs.append(msg.value().decode('utf-8'))
        print(f"🆕 NOVA: {msgs[-1][:100]}")

consumer.close()

if msgs:
    print(f"✅ {len(msgs)} novas mensagens recebidas!")
else:
    print("❌ Nenhuma nova mensagem em 10s")

⏳ Aguardando 10s por novas mensagens...


%6|1775830667.213|GETSUBSCRIPTIONS|rdkafka#consumer-3| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to eUpe9niRTiGd6hSIHQeDAw


🆕 NOVA: {"order_id": "ord_00240", "customer_id": "cust_016", "amount": 363.04, "currency": "BRL", "status": 
🆕 NOVA: {"order_id": "ord_00241", "customer_id": "cust_006", "amount": 238.62, "currency": "BRL", "status": 
🆕 NOVA: {"order_id": "ord_00242", "customer_id": "cust_014", "amount": 1207.35, "currency": "BRL", "status":
🆕 NOVA: {"order_id": "ord_00243", "customer_id": "cust_013", "amount": 23.15, "currency": "BRL", "status": "
🆕 NOVA: {"order_id": "ord_00244", "customer_id": "cust_019", "amount": 1273.44, "currency": "BRL", "status":
🆕 NOVA: {"order_id": "ord_00245", "customer_id": "cust_013", "amount": 1122.18, "currency": "BRL", "status":
🆕 NOVA: {"order_id": "ord_00246", "customer_id": "cust_010", "amount": 1014.42, "currency": "BRL", "status":
🆕 NOVA: {"order_id": "ord_00247", "customer_id": "cust_004", "amount": 781.77, "currency": "BRL", "status": 
🆕 NOVA: {"order_id": "ord_00248", "customer_id": "cust_017", "amount": 165.3, "currency": "BRL", "status": "
🆕 NOVA: {"order_id"